# Track1039: controlled quality-data trial

Attach PRIVATE **VGEC26-quality-track1039-private-20261019.zip** alone. Enable Internet and **GPU T4 x2**. Repository-read GITHUB_TOKEN is required; HF_TOKEN is optional, WANDB_API_KEY enables monitoring. Save Version / Run All. Dataset extraction names may vary. No notebook toggles or older notebook output needed.

The committed recipe fits Sailor2-20B classification and Qwen2.5-14B correction on the unchanged quality dataset, **NF4 QLoRA / FP16 compute, five FULL epochs each**. Same78 development rows,2034 private inputs inference-only, seed42, no folds/hints/decoder sweep. It exports four independent ZIPs: exact frozen quality control, classifier-only change, corrector-only change, both challengers. The small development set does not automatically choose a private winner.

Download the desired ZIP from the labeled links. Root results.zip changes only classification; both_challengers/results.zip uses both new models. comparison_metrics.json compares all four on the same development rows. Retain the whole output, including selected model bundles. Root W&B run stays alive through export. The owned temporary cache retains one base at a time, so stages may redownload weights; allow about40GiB free cache space. Native dual-T4 training and runtime still require verification.

The reported0.5806 quality result belongs to track1038. quality_control/results.zip is its exact original output; its same-track1039 score is unverified. The original quality run, source archive and reporting documents remain preserved.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
REPO_URL = "https://github.com/rquq/GECC_VLSP_26.git"
BRANCH = "kaggle-linux"
SOURCE_ROOT = Path("/kaggle/working")
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Fetch the committed source securely into a fresh directory.
from urllib.parse import urlsplit
import tempfile

with tempfile.TemporaryDirectory() as authentication:
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    token = None
    if urlsplit(REPO_URL).hostname == "github.com":
        from kaggle_secrets import UserSecretsClient
        try:
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            raise RuntimeError("Enable GITHUB_TOKEN in Kaggle Secrets with repository read access.") from None
        if not token:
            raise RuntimeError("GITHUB_TOKEN is empty; add a repository-read token in Kaggle Secrets.")
        askpass = Path(authentication) / "git-askpass.sh"
        askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) printf "%s\\n" x-access-token ;;\n'
                           '*) printf "%s\\n" "$GECC_GITHUB_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        env.update(GIT_ASKPASS=str(askpass), GECC_GITHUB_TOKEN=token)
    PROJECT_DIR = Path(tempfile.mkdtemp(prefix="gecc-source-", dir=SOURCE_ROOT))
    result = subprocess.run(["git", "-c", "credential.helper=", "clone", "--depth", "1",
                             "--single-branch", "--branch", BRANCH, "--", REPO_URL, str(PROJECT_DIR)],
                            env=env, capture_output=True, text=True)
    returncode = result.returncode
    detail = result.stderr.replace(token, "[REDACTED]") if token else result.stderr
    del env, token, result
    if returncode:
        raise RuntimeError("Cannot clone source; check Internet, branch, and GITHUB_TOKEN access. " + detail.strip())
SOURCE_INFO = {"repo_url": REPO_URL, "branch": BRANCH,
               "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip()}
print(json.dumps(SOURCE_INFO, indent=2))


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("gecc_notebook", PROJECT_DIR / "tools/notebook.py")
launcher = importlib.util.module_from_spec(spec)
spec.loader.exec_module(launcher)
TRAINING_PYTHON = launcher.setup_kaggle(PROJECT_DIR)


In [ ]:
RUN_DIR = launcher.launch(PROJECT_DIR, TRAINING_PYTHON)
